# 3 — ASD prevalence metrics by sex and age (OECD cohort)

## Objective

Derive analytical metrics from the processed OECD ASD prevalence dataset
to support comparison by gender, age range, region and time.

Main steps:

1. Load processed and auxiliary datasets
2. Merge regional classification data
3. Validate analysis-ready dataset
4. Compute gender- and age-based metrics
5. Prepare outputs for analysis and visualization

## 3.1 Environment and setup

Prepare the notebook environment for analysis.

Includes:
- core library imports
- environment validation

In [45]:
# Import core libraries and project paths for data access and analysis

import pandas as pd
import sys

from src.paths import INTERIM_DIR, PROCESSED_DIR

In [46]:
# Validate environment configuration and library availability for analysis

print("Python version:", sys.version.split()[0])

try:
    import src
    print("Project package import: OK")
except ImportError:
    print("ERROR: project package 'src' not found")

print("pandas version:", pd.__version__)

Python version: 3.13.5
Project package import: OK
pandas version: 2.2.3


## 3.2 Data ingestion

Load processed ASD prevalence data and auxiliary regional classification dataset.

Steps:
- load processed analytical dataset
- load country-to-region mapping dataset

In [47]:
# Load processed ASD prevalence dataset and validate structure

processed_path = PROCESSED_DIR / "gbd_asd_prevalence_oecd_processed.csv"
assert processed_path.exists(), f"Processed dataset not found: {processed_path}"

df = pd.read_csv(processed_path)

print("Shape:", df.shape)

# Inspect sample of processed dataset
df.head()

Shape: (38760, 7)


,country,gender,age_range,year,prevalence,upper_ui,lower_ui
0,Slovenia,Male,<5,1991,939.037586,1855.692963,459.595189
1,Slovenia,Female,<5,1991,363.753608,718.021074,177.685811
2,Slovenia,Male,5-9,1991,920.239063,1984.293942,435.700531
3,Slovenia,Female,5-9,1991,356.527957,772.392177,168.000119
4,Slovenia,Male,10-14,1991,912.201931,1979.073395,358.173398


In [48]:
# Load country-to-region mapping dataset and validate structure

regions_path = INTERIM_DIR / "country_regions_oecd.csv"
assert regions_path.exists(), f"Regions dataset not found: {regions_path}"

regions = pd.read_csv(regions_path)

print("Shape:", regions.shape)

# Inspect sample of regions dataset
regions.head()

Shape: (38, 2)


,country,region
0,Germany,Europe
1,Australia,Asia-Pacific
2,Austria,Europe
3,Belgium,Europe
4,Canada,North America


In [49]:
# Merge processed dataset with regional classification and validate mapping completeness

df = df.merge(regions, on="country", how="left")

print("Shape:", df.shape)

missing_regions = df["region"].isnull().sum()
print("\nMissing region values:", missing_regions)

assert missing_regions == 0, "Some countries are missing region mapping"

# Inspect sample after merge
df.head()

Shape: (38760, 8)

Missing region values: 0


,country,gender,age_range,year,prevalence,upper_ui,lower_ui,region
0,Slovenia,Male,<5,1991,939.037586,1855.692963,459.595189,Europe
1,Slovenia,Female,<5,1991,363.753608,718.021074,177.685811,Europe
2,Slovenia,Male,5-9,1991,920.239063,1984.293942,435.700531,Europe
3,Slovenia,Female,5-9,1991,356.527957,772.392177,168.000119,Europe
4,Slovenia,Male,10-14,1991,912.201931,1979.073395,358.173398,Europe


## 3.3 Metrics computation

Compute analytical metrics to compare ASD prevalence by gender across age ranges, time, countries and regions.

Steps:

1. Compute gender- and age-based metrics
2. Analyze temporal trends
3. Compare prevalence across countries
4. Compare prevalence across regions
5. Export analytical datasets for downstream analysis

### 3.3.1 Global metrics (gender × age_range)

Compare ASD prevalence by gender across age groups.

Steps:
- compute average prevalence by gender and age range
- compare prevalence across age groups

In [50]:
# Compute average prevalence by gender and age range and validate aggregated dataset

df_gender_age = df.groupby(["gender", "age_range"], as_index=False, sort=False).agg(prevalence=("prevalence", "mean"))

# Validate structure and completeness of gender-age aggregated dataset
print("Shape:", df_gender_age.shape)

print("\nGender categories:")
print(df_gender_age["gender"].unique())

print("\nAge range categories:")
print(df_gender_age["age_range"].unique())

print("\nDuplicate rows (gender-age_range):")
print(df_gender_age.duplicated(subset=["gender", "age_range"]).sum())

# Inspect sample of aggregated dataset
df_gender_age.head()

Shape: (30, 3)

Gender categories:
['Male' 'Female']

Age range categories:
['<5' '5-9' '10-14' '15-19' '20-24' '25-29' '30-34' '35-39' '40-44'
 '45-49' '50-54' '55-59' '60-64' '65-69' '70+']

Duplicate rows (gender-age_range):
0


,gender,age_range,prevalence
0,Male,<5,1511.793289
1,Female,<5,576.661732
2,Male,5-9,1437.185333
3,Female,5-9,537.343805
4,Male,10-14,1370.835216


In [51]:
# Build gender-age comparison dataset and compute metrics for cross-group analysis

age_order = ["<5","5-9","10-14","15-19","20-24","25-29","30-34","35-39",
             "40-44","45-49","50-54","55-59","60-64","65-69","70+"]

df_gender_age_pivot = (
    df_gender_age
    .pivot(index="age_range", columns="gender", values="prevalence")
    .rename_axis(None, axis=1)
    .reset_index()
)

# Standardize column names for consistent metric computation
df_gender_age_pivot = df_gender_age_pivot.rename(columns={"Female": "female_prevalence", "Male": "male_prevalence"})

# Reapply categorical order after pivot
df_gender_age_pivot["age_range"] = pd.Categorical(df_gender_age_pivot["age_range"], categories=age_order, ordered=True)
df_gender_age_pivot = df_gender_age_pivot.sort_values("age_range")

# Compute gender comparison metrics
df_gender_age_pivot["difference"] = df_gender_age_pivot["male_prevalence"] - df_gender_age_pivot["female_prevalence"]
df_gender_age_pivot["ratio"] = df_gender_age_pivot["male_prevalence"] / df_gender_age_pivot["female_prevalence"]

# Convert to percentage (rate per 100,000 → %)
df_gender_age_pivot["female_pct"] = df_gender_age_pivot["female_prevalence"] / 1000
df_gender_age_pivot["male_pct"] = df_gender_age_pivot["male_prevalence"] / 1000

# Apply rounding only after all computations
df_gender_age_pivot["female_prevalence"] = df_gender_age_pivot["female_prevalence"].round(0)
df_gender_age_pivot["male_prevalence"] = df_gender_age_pivot["male_prevalence"].round(0)
df_gender_age_pivot["difference"] = df_gender_age_pivot["difference"].round(0)
df_gender_age_pivot["ratio"] = df_gender_age_pivot["ratio"].round(2)
df_gender_age_pivot["female_pct"] = df_gender_age_pivot["female_pct"].round(2)
df_gender_age_pivot["male_pct"] = df_gender_age_pivot["male_pct"].round(2)

In [52]:
# Validate structure and completeness of gender-age comparison dataset

print("Shape:", df_gender_age_pivot.shape)

print("\nColumns:")
print(df_gender_age_pivot.columns)

print("\nMissing values:")
print(df_gender_age_pivot.isnull().sum())

Shape: (15, 7)

Columns:
Index(['age_range', 'female_prevalence', 'male_prevalence', 'difference',
       'ratio', 'female_pct', 'male_pct'],
      dtype='object')

Missing values:
age_range            0
female_prevalence    0
male_prevalence      0
difference           0
ratio                0
female_pct           0
male_pct             0
dtype: int64


In [53]:
# Inspect sample rows to verify metric behavior by gender and age range
df_gender_age_pivot.head()

,age_range,female_prevalence,male_prevalence,difference,ratio,female_pct,male_pct
14,<5,577.0,1512.0,935.0,2.62,0.58,1.51
8,5-9,537.0,1437.0,900.0,2.67,0.54,1.44
0,10-14,499.0,1371.0,872.0,2.75,0.50,1.37
1,15-19,465.0,1313.0,848.0,2.82,0.46,1.31
2,20-24,436.0,1264.0,829.0,2.90,0.44,1.26


### 3.3.2 Temporal trends (gender × year)

Analyze temporal evolution of ASD prevalence and compare trends between genders.

Steps:
- compute average prevalence by gender and year
- compare prevalence trends over time

In [54]:
# Compute average ASD prevalence by gender and year and validate temporal aggregation

df_gender_year = (
    df
    .groupby(["gender", "year"], as_index=False, sort=False)
    .agg(
        prevalence=("prevalence", "mean"),
        upper_ui=("upper_ui", "mean"),
        lower_ui=("lower_ui", "mean")
    )
)

print("Shape:", df_gender_year.shape)

print("\nGender categories:")
print(df_gender_year["gender"].unique())

print("\nYear range:")
print(df_gender_year["year"].min(), "-", df_gender_year["year"].max())

print("\nDuplicate rows (gender-year):")
print(df_gender_year.duplicated(subset=["gender", "year"]).sum())

# Inspect sample of aggregated dataset
df_gender_year.head()

Shape: (68, 5)

Gender categories:
['Male' 'Female']

Year range:
1990 - 2023

Duplicate rows (gender-year):
0


,gender,year,prevalence,upper_ui,lower_ui
0,Male,1991,1064.618783,2286.145529,445.576967
1,Female,1991,338.550041,668.110457,160.183593
2,Male,1990,1062.936598,2294.881806,440.058161
3,Female,1990,337.257827,668.962953,158.310541
4,Male,1992,1066.683471,2278.685534,450.429361


In [55]:
# Build gender-year comparison dataset and compute temporal metrics

df_gender_year_pivot = (
    df_gender_year
    .pivot(index="year", columns="gender", values="prevalence")
    .rename_axis(None, axis=1)
    .reset_index()
)

# Standardize column names for consistent metric computation
df_gender_year_pivot = df_gender_year_pivot.rename(columns={"Female": "female_prevalence", "Male": "male_prevalence"})

# Compute gender comparison metrics
df_gender_year_pivot["difference"] = df_gender_year_pivot["male_prevalence"] - df_gender_year_pivot["female_prevalence"]
df_gender_year_pivot["ratio"] = df_gender_year_pivot["male_prevalence"] / df_gender_year_pivot["female_prevalence"]

# Convert to percentage (rate per 100,000 → %)
df_gender_year_pivot["female_pct"] = df_gender_year_pivot["female_prevalence"] / 1000
df_gender_year_pivot["male_pct"] = df_gender_year_pivot["male_prevalence"] / 1000

# Apply rounding only after all computations
df_gender_year_pivot["female_prevalence"] = df_gender_year_pivot["female_prevalence"].round(0)
df_gender_year_pivot["male_prevalence"] = df_gender_year_pivot["male_prevalence"].round(0)
df_gender_year_pivot["difference"] = df_gender_year_pivot["difference"].round(0)
df_gender_year_pivot["ratio"] = df_gender_year_pivot["ratio"].round(2)
df_gender_year_pivot["female_pct"] = df_gender_year_pivot["female_pct"].round(2)
df_gender_year_pivot["male_pct"] = df_gender_year_pivot["male_pct"].round(2)

# Ensure temporal ordering for consistent trend analysis
df_gender_year_pivot = df_gender_year_pivot.sort_values("year").reset_index(drop=True)

In [56]:
# Validate structure and completeness of gender-year comparison dataset

print("Shape:", df_gender_year_pivot.shape)

print("\nColumns:")
print(df_gender_year_pivot.columns)

print("\nYear range:")
print(df_gender_year_pivot["year"].min(), "-", df_gender_year_pivot["year"].max())

print("\nDuplicate rows (year):")
print(df_gender_year_pivot.duplicated(subset=["year"]).sum())

Shape: (34, 7)

Columns:
Index(['year', 'female_prevalence', 'male_prevalence', 'difference', 'ratio',
       'female_pct', 'male_pct'],
      dtype='object')

Year range:
1990 - 2023

Duplicate rows (year):
0


In [57]:
# Inspect sample rows to verify temporal metric behavior

df_gender_year_pivot.head()

,year,female_prevalence,male_prevalence,difference,ratio,female_pct,male_pct
0,1990,337.0,1063.0,726.0,3.15,0.34,1.06
1,1991,339.0,1065.0,726.0,3.14,0.34,1.06
2,1992,340.0,1067.0,727.0,3.14,0.34,1.07
3,1993,342.0,1069.0,728.0,3.13,0.34,1.07
4,1994,343.0,1072.0,729.0,3.12,0.34,1.07


### 3.3.3 Country comparison (country × gender)

Compare ASD prevalence across OECD countries and identify differences between genders.

Steps:
- compute average prevalence by country and gender
- compare prevalence across countries

In [58]:
# Compute average prevalence by country and gender to support cross-country comparison

df_country_gender = df.groupby(["country", "gender"], as_index=False, sort=False).agg(prevalence=("prevalence", "mean"))

# Validate aggregation structure
print("Shape:", df_country_gender.shape)

print("\nGender categories:")
print(df_country_gender["gender"].unique())

print("\nCountries:", df_country_gender["country"].nunique())

print("\nDuplicate rows (country-gender):")
print(df_country_gender.duplicated(subset=["country", "gender"]).sum())

# Inspect sample
df_country_gender.head()

Shape: (76, 3)

Gender categories:
['Male' 'Female']

Countries: 38

Duplicate rows (country-gender):
0


,country,gender,prevalence
0,Slovenia,Male,851.153804
1,Slovenia,Female,323.760190
2,Turkey,Male,724.411452
3,Turkey,Female,295.322775
4,Chile,Male,1232.814387


In [59]:
# Build country-level gender comparison dataset and compute metrics

df_country_gender_pivot = (
    df_country_gender
    .pivot(index="country", columns="gender", values="prevalence")
    .rename_axis(None, axis=1)
    .reset_index()
)

# Standardize column names for consistent metric computation
df_country_gender_pivot = df_country_gender_pivot.rename(columns={"Female": "female_prevalence", "Male": "male_prevalence"})

# Compute gender comparison metrics
df_country_gender_pivot["difference"] = (df_country_gender_pivot["male_prevalence"]- df_country_gender_pivot["female_prevalence"])
df_country_gender_pivot["ratio"] = (df_country_gender_pivot["male_prevalence"] / df_country_gender_pivot["female_prevalence"])

# Convert to percentage (rate per 100,000 → %)
df_country_gender_pivot["female_pct"] = df_country_gender_pivot["female_prevalence"] / 1000
df_country_gender_pivot["male_pct"] = df_country_gender_pivot["male_prevalence"] / 1000

# Apply rounding only after all computations
df_country_gender_pivot["female_prevalence"] = df_country_gender_pivot["female_prevalence"].round(0)
df_country_gender_pivot["male_prevalence"] = df_country_gender_pivot["male_prevalence"].round(0)
df_country_gender_pivot["difference"] = df_country_gender_pivot["difference"].round(0)
df_country_gender_pivot["ratio"] = df_country_gender_pivot["ratio"].round(2)
df_country_gender_pivot["female_pct"] = df_country_gender_pivot["female_pct"].round(2)
df_country_gender_pivot["male_pct"] = df_country_gender_pivot["male_pct"].round(2)

In [60]:
# Validate structure and completeness of country-level comparison dataset

print("Shape:", df_country_gender_pivot.shape)

print("\nColumns:")
print(df_country_gender_pivot.columns)

print("\nDuplicate rows (country):")
print(df_country_gender_pivot.duplicated(subset=["country"]).sum())

Shape: (38, 7)

Columns:
Index(['country', 'female_prevalence', 'male_prevalence', 'difference',
       'ratio', 'female_pct', 'male_pct'],
      dtype='object')

Duplicate rows (country):
0


In [61]:
# Inspect sample rows to verify country-level metric behavior

df_country_gender_pivot.head()

,country,female_prevalence,male_prevalence,difference,ratio,female_pct,male_pct
0,Australia,448.0,1303.0,856.0,2.91,0.45,1.30
1,Austria,349.0,1181.0,832.0,3.39,0.35,1.18
2,Belgium,341.0,1152.0,811.0,3.38,0.34,1.15
3,Canada,518.0,1402.0,884.0,2.70,0.52,1.40
4,Chile,418.0,1233.0,815.0,2.95,0.42,1.23


### 3.3.4 Regional comparison (region × gender)

Compare ASD prevalence across regions and identify differences between genders.

Steps:
- compute average prevalence by region and gender
- compare prevalence across regions

In [62]:
# Compute average prevalence by region and gender 
df_region_gender = df.groupby(["region", "gender"], as_index=False, sort=False).agg(prevalence=("prevalence", "mean"))

# Validate aggregation structure
print("Shape:", df_region_gender.shape)

print("\nGender categories:")
print(df_region_gender["gender"].unique())

print("\nRegions:", df_region_gender["region"].unique())

print("\nDuplicate rows (region-gender):")
print(df_region_gender.duplicated(subset=["region", "gender"]).sum())

# Inspect sample of aggregated dataset
df_region_gender.head()

Shape: (10, 3)

Gender categories:
['Male' 'Female']

Regions: ['Europe' 'Middle East' 'Latin America' 'Asia-Pacific' 'North America']

Duplicate rows (region-gender):
0


,region,gender,prevalence
0,Europe,Male,1122.493275
1,Europe,Female,356.672457
2,Middle East,Male,911.317815
3,Middle East,Female,318.711392
4,Latin America,Male,1010.108761


In [63]:
# Build region-level gender comparison dataset and compute metrics

df_region_gender_pivot = (
    df_region_gender
    .pivot(index="region", columns="gender", values="prevalence")
    .rename_axis(None, axis=1)
    .reset_index()
)

# Standardize column names for consistent metric computation
df_region_gender_pivot = df_region_gender_pivot.rename(columns={"Female": "female_prevalence", "Male": "male_prevalence"})

# Compute gender comparison metrics
df_region_gender_pivot["difference"] = (df_region_gender_pivot["male_prevalence"] - df_region_gender_pivot["female_prevalence"])
df_region_gender_pivot["ratio"] = (df_region_gender_pivot["male_prevalence"] / df_region_gender_pivot["female_prevalence"])

# Convert to percentage (rate per 100,000 → %)
df_region_gender_pivot["female_pct"] = df_region_gender_pivot["female_prevalence"] / 1000
df_region_gender_pivot["male_pct"] = df_region_gender_pivot["male_prevalence"] / 1000

# Apply rounding only after all computations
df_region_gender_pivot["female_prevalence"] = df_region_gender_pivot["female_prevalence"].round(0)
df_region_gender_pivot["male_prevalence"] = df_region_gender_pivot["male_prevalence"].round(0)
df_region_gender_pivot["difference"] = df_region_gender_pivot["difference"].round(0)
df_region_gender_pivot["ratio"] = df_region_gender_pivot["ratio"].round(2)
df_region_gender_pivot["female_pct"] = df_region_gender_pivot["female_pct"].round(2)
df_region_gender_pivot["male_pct"] = df_region_gender_pivot["male_pct"].round(2)

In [64]:
# Validate structure and completeness of region-level comparison dataset

print("Shape:", df_region_gender_pivot.shape)

print("\nColumns:")
print(df_region_gender_pivot.columns)

print("\nDuplicate rows (region):")
print(df_region_gender_pivot.duplicated(subset=["region"]).sum())

Shape: (5, 7)

Columns:
Index(['region', 'female_prevalence', 'male_prevalence', 'difference', 'ratio',
       'female_pct', 'male_pct'],
      dtype='object')

Duplicate rows (region):
0


In [65]:
# Inspect sample rows to verify region-level metric behavior

df_region_gender_pivot.head()

,region,female_prevalence,male_prevalence,difference,ratio,female_pct,male_pct
0,Asia-Pacific,539.0,1472.0,933.0,2.73,0.54,1.47
1,Europe,357.0,1122.0,766.0,3.15,0.36,1.12
2,Latin America,366.0,1010.0,644.0,2.76,0.37,1.01
3,Middle East,319.0,911.0,593.0,2.86,0.32,0.91
4,North America,459.0,1200.0,741.0,2.62,0.46,1.20


### 3.3.5 Export analytical datasets

Export computed analytical datasets to the processed data layer for reuse in downstream analysis and visualization.

In [66]:
# Export analytical datasets for downstream analysis and visualization

df_gender_age_pivot.to_csv(PROCESSED_DIR / "df_gender_age_pivot.csv", index=False)
df_gender_year_pivot.to_csv(PROCESSED_DIR / "df_gender_year_pivot.csv", index=False)
df_country_gender_pivot.to_csv(PROCESSED_DIR / "df_country_gender_pivot.csv", index=False)
df_region_gender_pivot.to_csv(PROCESSED_DIR / "df_region_gender_pivot.csv", index=False)

print("Saved:", PROCESSED_DIR / "df_gender_age_pivot.csv")
print("Saved:", PROCESSED_DIR / "df_gender_year_pivot.csv")
print("Saved:", PROCESSED_DIR / "df_country_gender_pivot.csv")
print("Saved:", PROCESSED_DIR / "df_region_gender_pivot.csv")

Saved: C:\Users\samai\OneDrive\Escritorio\PROYECTO AUTISMO\autism-diagnosis-gender-gap\data\3_processed\df_gender_age_pivot.csv
Saved: C:\Users\samai\OneDrive\Escritorio\PROYECTO AUTISMO\autism-diagnosis-gender-gap\data\3_processed\df_gender_year_pivot.csv
Saved: C:\Users\samai\OneDrive\Escritorio\PROYECTO AUTISMO\autism-diagnosis-gender-gap\data\3_processed\df_country_gender_pivot.csv
Saved: C:\Users\samai\OneDrive\Escritorio\PROYECTO AUTISMO\autism-diagnosis-gender-gap\data\3_processed\df_region_gender_pivot.csv
